## Stolen Base Report 
Run the code start to finish. Only the game configuration cell and the out path need to be edited.

In [30]:
import numpy as np, pandas as pd, os

from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT

In [31]:
weight_pop_time = 0.50
weight_throw_speed = 0.30
weight_exchange_time = 0.20
min_att = 6

def compute_backstop_plus(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.dropna(subset=["PopTime", "ExchangeTime", "ThrowSpeed"]).copy()

    lg_pop_mean, lg_pop_std = df["PopTime"].mean(), df["PopTime"].std()
    lg_exch_mean, lg_exch_std = df["ExchangeTime"].mean(), df["ExchangeTime"].std()
    lg_speed_mean, lg_speed_std = df["ThrowSpeed"].mean(), df["ThrowSpeed"].std()

    agg = df.groupby("Catcher").agg(
        pop_time = ("PopTime", "mean"),
        exchange_time = ("ExchangeTime", "mean"),
        throw_speed = ("ThrowSpeed", "mean"),
        attempts = ("PopTime", "count")
    ).reset_index()

    agg["z_pop"] = (lg_pop_mean - agg["pop_time"]) / lg_pop_std
    agg["z_exchange"] = (lg_exch_mean - agg["exchange_time"]) / lg_exch_std
    agg["z_speed"] = (agg["throw_speed"] - lg_speed_mean) / lg_speed_std

    agg["composite_z"] = (
        weight_pop_time * agg["z_pop"]
        + weight_throw_speed * agg["z_speed"]
        + weight_exchange_time * agg["z_exchange"]
    )
    agg["backstop_plus"] = (100 + agg["composite_z"] * 15).round(1)

    agg = agg.sort_values("backstop_plus", ascending = False).reset_index(drop = True)
    agg.attrs["league_exchange_mean"] = lg_exch_mean
    agg.attrs["league_throw_speed_mean"] = lg_speed_mean
    return agg

def backstop_tier(backstop_plus: float) -> str:
    if backstop_plus >= 115:
        return "Elite"
    if backstop_plus >= 105:
        return "Above Average"
    if backstop_plus >= 95:
        return "Average"
    if backstop_plus >= 85:
        return "Below Average"
    return "Weak"
    

In [32]:
baseline_prob = {
    "Fastball": 0.682,
    "Breaking Ball": 0.788,
    "Off-Speed": 0.681,
}

coef_throw_speed = -0.0764
coef_exchange_time = 1.2822
coef_pitcher_throws_right = -0.4183
count_coef = {
    "0-0": 0.0,
    "Hitter Ahead": -0.5348,
    "Even": -0.6455,
    "Pitcher Ahead": -0.4081,
    "Full": -1.6789,
}
baseline_count = "Even"

mlg_throw_speed_mean = 77.302
mlg_exchange_time_mean = 0.795

def _logit(p):
    return np.log(p / (1-p))

def _inv_logit(x):
    return 1 / (1 + np.exp(-x))

def adjusted_steal_probability(
    pitch_category: str,
    count: str,
    pitcher_throws: str,
    catcher_throw_speed: float,
    catcher_exchange_time: float,
    league_throw_speed_mean: float = mlg_throw_speed_mean,
    league_exchange_time_mean: float = mlg_exchange_time_mean,
) -> float:
    baseline = baseline_prob[pitch_category]
    logit_p = _logit(baseline)
    logit_p += coef_throw_speed * (catcher_throw_speed - league_throw_speed_mean)
    logit_p += coef_exchange_time * (catcher_exchange_time - league_exchange_time_mean)

    if pitcher_throws.strip().lower().startswith("l"):
        logit_p += -coef_pitcher_throws_right

    logit_p += count_coef[count] - count_coef[baseline_count]

    return float(_inv_logit(logit_p))

def traffic_light(prob: float) -> str:
    if prob >= 0.75:
        return "green"
    if prob >= 0.65:
        return "yellow"
    return "red"


In [33]:
green = colors.HexColor("#2E7D32")
green_bg = colors.HexColor("#C8E6C9")
yellow = colors.HexColor("#F9A825")
yellow_bg = colors.HexColor("#FFF3C4")
red = colors.HexColor("#C62828")
red_bg = colors.HexColor("#FFCDD2")
navy = colors.HexColor("#1A2340")

TIER_COLOR = {
    "Elite": red,
    "Above Average": red,
    "Average": yellow,
    "Below Average": green,
    "Weak": green,
}

pitch_categories = ["Fastball", "Breaking Ball", "Off-Speed"]
counts = ["0-0", "Hitter Ahead", "Even", "Pitcher Ahead", "Full"]

def _cell_color(prob): 
    light = traffic_light(prob)
    return {"green": green_bg, "yellow": yellow_bg, "red": red_bg}[light]

def pitch_type_label(cat, pitch_count_by_type):
    total = sum(pitch_count_by_type.values())
    count = pitch_count_by_type.get(cat, 0)
    pct = round((count / total) * 100, 1) if total else 0.0
    return f"{cat} ({count}, {pct}%)"

def build_grid(catcher, pitcher_throws):
    grid = {}
    for cat in pitch_categories:
        grid[cat] = {}
        for count in counts:
            grid[cat][count] = adjusted_steal_probability(
                pitch_category = cat,
                count = count,
                pitcher_throws = pitcher_throws,
                catcher_throw_speed = catcher["throw_speed"],
                catcher_exchange_time = catcher["exchange_time"],
            )
    return grid

def blended_probs_by_count(grid, pitch_count_by_type):
    total = sum(pitch_count_by_type.values())
    mix_pct = {
        cat: (pitch_count_by_type.get(cat, 0) / total if total else 0)
        for cat in pitch_categories
    }
    return {
        count: sum(mix_pct[cat] * grid[cat][count] for cat in pitch_categories)
        for count in counts
    }

def best_and_worst(blended):
    best = max(blended.items(), key = lambda x: x[1])
    worst = min(blended.items(), key = lambda x: x[1])
    return best, worst

def build_pdf(out_path, team_name, opponent_name, game_date, 
              opposing_pitcher_name, opposing_pitcher_throws, 
              catcher, grid, best, worst, pitch_count_by_type,
             catcher_sba, catcher_rcs, catcher_rcs_pct):
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "TitleBig", parent=styles["Title"], fontSize=22, textColor=navy,
        alignment=TA_CENTER, spaceAfter=2,
    )
    subtitle_style = ParagraphStyle(
        "Subtitle", parent=styles["Normal"], fontSize=12, textColor=colors.grey,
        alignment=TA_CENTER, spaceAfter=10,
    )
    section_style = ParagraphStyle(
        "Section", parent=styles["Heading2"], fontSize=13, textColor=navy,
        spaceBefore=10, spaceAfter=4,
    )
    body_style = ParagraphStyle("Body", parent=styles["Normal"], fontSize=11, leading=15)
    callout_style = ParagraphStyle(
        "Callout", parent=styles["Normal"], fontSize=13, leading=17,
        textColor=colors.white, alignment=TA_LEFT,
    )

    doc = SimpleDocTemplate(
        out_path, pagesize=landscape(letter),
        topMargin=0.4 * inch, bottomMargin=0.4 * inch,
        leftMargin=0.5 * inch, rightMargin=0.5 * inch,
    )
    story = []

    story.append(Paragraph("STOLEN BASE PREGAME REPORT", title_style))
    story.append(Paragraph(
        f"{team_name} vs. {opponent_name} &nbsp;|&nbsp; {game_date}", subtitle_style,
    ))
    story.append(HRFlowable(width = "100%", color = navy, thickness = 1.2))

    tier = backstop_tier(catcher["backstop_plus"]) if catcher["backstop_plus"] is not None else "Unknown"
    tier_color = TIER_COLOR.get(tier, colors.grey)
    sample_note = " (small sample -- treat as directional)" if catcher.get("low_sample") else ""
    if catcher["backstop_plus"] is None:
        catcher_line = f"<b>{catcher['name']}</b> -- no Backstop+ history yet. Using scouted arm strength only."
    else:
        catcher_line = (
    f"<b>{catcher['name']}</b> &nbsp; | &nbsp; "
    f"Backstop+ <b>{catcher['backstop_plus']}</b> &nbsp; | &nbsp; "
    f"<font color='{tier_color.hexval()}'><b>{tier.upper()}</b></font>"
    )
    if catcher.get("low_sample"):
        catcher_line += (
            f" &nbsp; <font color='{red.hexval()}'>"
            f"<b> ! SMALL SAMPLE (ONLY {catcher['attempts']} ATTEMPTS) !</b></font>"
        )
    def _fmt_stat(value):
        return "N/A" if value is None else f"{value}"
        
    def _fmt_pct(value):
        return "N/A" if value is None else f"{value*100:.1f}%"

    stats_line = (
        f"SBA: <b>{_fmt_stat(catcher_sba)}</b> &nbsp; | &nbsp; "
        f"RCS: <b>{_fmt_stat(catcher_rcs)}</b> &nbsp; | &nbsp; "
        f"RCS%: <b>{_fmt_pct(catcher_rcs_pct)}</b>"
    )
    
    pitcher_line = f"Opposing Pitcher: <b>{opposing_pitcher_name}</b> ({opposing_pitcher_throws}-Handed)"
    catcher_box_data = [
        [Paragraph("OPPOSING CATCHER", ParagraphStyle("lbl", parent=body_style, fontSize=9, textColor=colors.grey))],
        [Paragraph(catcher_line, ParagraphStyle("cl", parent=body_style, fontSize=14))],
        [Paragraph(stats_line, body_style)],
        [Paragraph(pitcher_line, body_style)],
    ]
    catcher_box = Table(catcher_box_data, colWidths=[10 * inch])
    catcher_box.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#F2F3F7")),
        ("BOX", (0, 0), (-1, -1), 1, colors.HexColor("#D0D3DE")),
        ("LEFTPADDING", (0, 0), (-1, -1), 12),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    story.append(Spacer(1, 8))
    story.append(catcher_box)

    story.append(Paragraph("STEAL PROBABILITY BY PITCH TYPE &amp; COUNT", section_style))
    header_row = ["Pitch Type (n, %)"] + counts
    table_data = [header_row]
    cell_colors = []
    for r, cat in enumerate(pitch_categories, start = 1):
        row = [pitch_type_label(cat, pitch_count_by_type)] 
        for c, count in enumerate(counts, start = 1):
            prob = grid[cat][count]
            row.append(f"{prob*100:.0f}%")
            cell_colors.append((r, c, _cell_color(prob)))
        table_data.append(row)

    col_widths = [2.5 * inch] + [1.5 * inch] * len(counts)
    grid_table = Table(table_data, colWidths = col_widths, rowHeights = 0.55 * inch)
    style_cmds = [
        ("BACKGROUND", (0, 0), (-1, 0), navy),
        ("TEXTCOLOR", (0,0), (-1, 0), colors.white),
        ("FONTNAME", (0,0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0,0), (-1, 0), 12),
        ("FONTNAME", (0, 1), (0, -1), "Helvetica-Bold"),
        ("FONTSIZE", (0, 1), (-1, -1), 16),
        ("FONTSIZE", (0, 1), (0, -1), 12),
        ("ALIGN", (1, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0,0), (-1, -1), "MIDDLE"),
        ("GRID", (0,0), (-1, -1), 1, colors.white),
        ("BOX", (0,0), (-1, -1), 1.5, navy),
        ("LEFTPADDING", (0,0), (0, -1), 10),
         ]
    for r, c, color in cell_colors:
        style_cmds.append(("BACKGROUND", (c, r), (c, r), color))
    grid_table.setStyle(TableStyle(style_cmds))
    story.append(Spacer(1, 6))
    story.append(grid_table)

    story.append(Spacer(1, 4))
    legend_text = (
        f"<font color='{green.hexval()}'><b>&#9632; GREEN</b></font> = green light &nbsp;&nbsp;&nbsp; "
        f"<font color='{yellow.hexval()}'><b>&#9632; YELLOW</b></font> = read it, run only on a great jump &nbsp;&nbsp;&nbsp; "
        f"<font color='{red.hexval()}'><b>&#9632; RED</b></font> = hold"
    )
    story.append(Paragraph(legend_text, ParagraphStyle("legend", parent = body_style, fontSize = 10, textColor = colors.grey)))

    best_count, best_p = best
    worst_count, worst_p = worst
    callout_text = (
        f"<b>BEST CHANCE:</b> {best_count} count ({best_p*100:.0f}% blended) "
        f"&nbsp;&nbsp;|&nbsp;&nbsp; <b>AVOID:</b> a {worst_count} count ({worst_p*100:.0f}% blended)"
    )
    callout_table = Table([[Paragraph(callout_text, callout_style)]], colWidths=[10 * inch])
    callout_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), navy),
        ("LEFTPADDING", (0, 0), (-1, -1), 14),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
    ]))
    story.append(Spacer(1, 10))
    story.append(callout_table)

    story.append(Spacer(1, 8))
    story.append(Paragraph(
        "Model: NECBL steal-probability regression (n ≈ 300, MAE ≈ 3.5 pts), baseline by pitch "
        "category adjusted for this catcher's actual throw speed &amp; exchange time, pitcher "
        "handedness, and count.<br/>"
        "Movement data (IVB/Horz Break) not used when unavailable pregame.<br/>"
        "Blended percentage = steal probability weighted by this pitcher's actual pitch mix "
        "(counts/percentages shown in the table above), giving the realistic odds for a given "
        "count before the pitch type is known.",
        ParagraphStyle("foot", parent=body_style, fontSize=8, textColor=colors.grey),
    ))

    doc.build(story)
    print(f"Saved: {out_path}")

### Game Configuration
The cell below is the only cell that needs to be edited on a game by game basis.
- Players must follow the format "Lastname, Firstname"
- For Pitcher Throws ("Right", "Left")
- Manual Catcher if none/bad data
- Enter RCS% as a decimal

In [47]:
team_name = "Bristol Blues"
opponent_name = "Danbury Westerners"
game_date = "2026-07-05"

file = "trackman_stolenbases.xlsx" #update weekly
sheet = "in"

opposing_pitcher_name = "Gibson, Cole"
opposing_pitcher_throws = "Right"
#Personalized Pitch Mix
pitch_count_by_type = {
    "Fastball": 172, 
    "Breaking Ball": 74, 
    "Off-Speed": 14,
}

opposing_catcher_name = "Mowry, Collin"

use_manual_catcher = False
manual_catcher_throw_speed = None
manual_catcher_exchange_time = None
manual_catcher_backstop_plus = None

opposing_catcher_sba = 18
opposing_catcher_rcs = 11
opposing_catcher_rcs_pct = .379 

In [49]:
def get_catcher_profile():
    if use_manual_catcher:
        if manual_catcher_throw_speed is None or manual_catcher_exchange_time is None:
            raise ValueError(
                "use_manual_catcher is True but throw speed and/or exchange time were not provided."
            )
        return {
            "name": opposing_catcher_name,
            "throw_speed": manual_catcher_throw_speed,
            "exchange_time": manual_catcher_exchange_time,
            "backstop_plus": manual_catcher_backstop_plus,
            "attempts": None,
            "low_sample": manual_catcher_backstop_plus is None,
        }

    raw = pd.read_excel(file, sheet_name = sheet)
    table = compute_backstop_plus(raw)
    match = table[table["Catcher"].str.strip().str.lower() == opposing_catcher_name.strip().lower()]

    if match.empty:
        raise ValueError(
            f"'{opposing_catcher_name}' not found in {file}. "
            f"Fix the name (must match 'Last, First') or set use_manual_catcher = True."
        )

    row = match.iloc[0]
    return {
        "name": row["Catcher"],
        "throw_speed": row["throw_speed"],
        "exchange_time": row["exchange_time"],
        "backstop_plus": row["backstop_plus"],
        "attempts": int(row["attempts"]),
        "low_sample": row["attempts"] < min_att,
    }


catcher = get_catcher_profile()
grid = build_grid(catcher, opposing_pitcher_throws)
blended = blended_probs_by_count(grid, pitch_count_by_type)
best, worst = best_and_worst(blended)

output_folder = os.path.join(os.path.expanduser("~"), "Desktop", "Bristol Blues", "SB Reports") #Customize Out Path
os.makedirs(output_folder, exist_ok = True)
out_path = os.path.join(output_folder, f"SB_Report_{opponent_name.replace(' ', '')}_{game_date}.pdf")

build_pdf(
    out_path = out_path,
    team_name = team_name,
    opponent_name = opponent_name,
    game_date = game_date,
    opposing_pitcher_name = opposing_pitcher_name,
    opposing_pitcher_throws = opposing_pitcher_throws,
    catcher = catcher,
    grid = grid,
    best = best,
    worst = worst,
    pitch_count_by_type = pitch_count_by_type,
    catcher_sba = opposing_catcher_sba,
    catcher_rcs = opposing_catcher_rcs,
    catcher_rcs_pct = opposing_catcher_rcs_pct,
)

Saved: /Users/kylesmith/Desktop/Bristol Blues/SB Reports/SB_Report_DanburyWesterners_2026-07-05.pdf
